# 06 · Camada Gold — Star Schema (Datamart)

Constrói o datamart em modelo dimensional (Star Schema) a partir de
`silver.incidentes_tratados` — **sem** as features de ML (lags, médias
móveis), como combinado: essas ficam só nas tabelas de série temporal da
Silver, que alimentam o treino dos modelos, não o BI executivo.

**Modelo dimensional:**

| Tabela | Tipo | Grão / Chave |
|---|---|---|
| `gold.dim_data` | Dimensão | 1 linha por dia (`data_sk` = yyyyMMdd), inclui `is_feriado`/`nome_feriado` |
| `gold.dim_produto` | Dimensão | 1 linha por produto (`produto_sk`) |
| `gold.dim_categoria` | Dimensão | 1 linha por categoria (`categoria_sk`) |
| `gold.dim_equipe` | Dimensão | 1 linha por grupo designado (`equipe_sk`) |
| `gold.dim_prioridade` | Dimensão | 1 linha por prioridade, já carrega o SLA (`prioridade_num`) |
| `gold.fato_incidentes` | Fato | 1 linha por incidente, FKs para as 5 dimensões |
| `gold.fato_incidentes_diario` | Fato agregada | 1 linha por (dia, produto, categoria, prioridade) — para Power BI via DirectQuery, sem precisar agregar 122k linhas em tempo real a cada refresh de dashboard. Inclui `qtd_kpi_violado` (bruto) e `taxa_violacao_kpi` (%), os dois juntos |

**Por que `Item de configuração` e `Subcategoria` não viraram dimensão
própria:** cardinalidade alta demais (9.171 e 447 valores distintos,
respectivamente — achados no notebook `04`), ficam como atributo
descritivo direto na fato (dimensão degenerada), evitando uma dimensão
quase do tamanho da própria fato.

In [ ]:
%run ./00_config

In [ ]:
from pyspark.sql import functions as F, Window
from datetime import timedelta

silver_incidentes = spark.table(qualified_table(SCHEMA_SILVER, "incidentes_tratados"))
print(f"Lendo silver.incidentes_tratados: {silver_incidentes.count()} linhas")

## Período do calendário — calculado, com buffer futuro

`DATA_MIN` = a mesma lógica do notebook `05` (não hardcoded, vem do
dado). `DATA_MAX`, diferente da Silver, **estende para o futuro** além
da última data real — a `dim_data` precisa ter linhas prontas para os
dias em que o modelo vai gerar previsão (D+1...D+7), senão a fato de
previsões (`gold.previsoes_incidentes`, ainda não criada) não teria uma
chave de data pra se relacionar. Isso é seguro aqui — e não seria na
Silver — porque `dim_data` é só atributo de calendário (dia da semana,
feriado...), não tem `qtd_incidentes` nenhuma pra "inventar".

In [ ]:
DIAS_BUFFER_FUTURO = 7  # cobre pelo menos o horizonte de previsão D+7

_periodo = silver_incidentes.agg(F.min("data_abertura").alias("min"), F.max("data_abertura").alias("max")).first()
DATA_MIN = _periodo["min"].isoformat()
_data_max_historica = _periodo["max"]
DATA_MAX = (_data_max_historica + timedelta(days=DIAS_BUFFER_FUTURO)).isoformat()

print(f"Período histórico: {DATA_MIN} a {_data_max_historica.isoformat()}")
print(f"dim_data estendida até: {DATA_MAX} (+{DIAS_BUFFER_FUTURO} dias além do último dado real)")

## Dimensões

In [ ]:
dim_data = (
    spark.range(1)
    .select(F.explode(F.sequence(F.lit(DATA_MIN).cast("date"), F.lit(DATA_MAX).cast("date"))).alias("data"))
    .withColumn("data_sk", F.date_format("data", "yyyyMMdd").cast("int"))
    .withColumn("ano", F.year("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("dia", F.dayofmonth("data"))
    .withColumn("trimestre", F.quarter("data"))
    .withColumn("dia_semana_num", F.dayofweek("data"))
    .withColumn("dia_semana_nome", F.date_format("data", "EEEE"))
    .withColumn("is_fim_de_semana", F.col("dia_semana_num").isin(1, 7))
    # nome_feriado vem de silver.calendario_feriados (tabela de referência,
    # não de silver.features_calendario — lá é só o flag numérico is_feriado,
    # dado categórico não é feature de modelo, ver notebook 05).
    .join(
        spark.table(qualified_table(SCHEMA_SILVER, "calendario_feriados"))
        .select(F.col("data_abertura").alias("data"), "nome_feriado"),
        "data",
        "left",
    )
    .withColumn("is_feriado", F.when(F.col("nome_feriado").isNotNull(), F.lit(1)).otherwise(F.lit(0)))
)

dim_produto = (
    silver_incidentes.select("produto").distinct()
    .withColumn("produto_sk", F.row_number().over(Window.orderBy("produto")))
    .select("produto_sk", "produto")
)

dim_categoria = (
    silver_incidentes.select("categoria").distinct()
    .withColumn("categoria_sk", F.row_number().over(Window.orderBy("categoria")))
    .select("categoria_sk", "categoria")
)

dim_equipe = (
    silver_incidentes.select("grupo_designado").distinct()
    .withColumn("equipe_sk", F.row_number().over(Window.orderBy("grupo_designado")))
    .select("equipe_sk", "grupo_designado")
)

dim_prioridade = silver_incidentes.select("prioridade_num", "prioridade_desc", "sla_segundos").distinct()

for nome, df in [
    ("dim_data", dim_data), ("dim_produto", dim_produto), ("dim_categoria", dim_categoria),
    ("dim_equipe", dim_equipe), ("dim_prioridade", dim_prioridade),
]:
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(qualified_table(SCHEMA_GOLD, nome)))
    print(f"gold.{nome} gravada: {df.count()} linhas")

## Fato — `gold.fato_incidentes`

Grão: 1 linha por incidente. Junta com as dimensões para trazer as
surrogate keys; mantém os atributos de cardinalidade alta
(`subcategoria`, `item_configuracao`) direto na fato.

In [ ]:
fato_incidentes = (
    silver_incidentes
    .join(dim_produto, "produto", "left")
    .join(dim_categoria, "categoria", "left")
    .join(dim_equipe, "grupo_designado", "left")
    .withColumn("data_sk", F.date_format("data_abertura", "yyyyMMdd").cast("int"))
    .select(
        "numero_incidente", "data_sk", "produto_sk", "categoria_sk", "equipe_sk", "prioridade_num",
        "subcategoria", "item_configuracao", "status", "aberto_por",
        "duracao_segundos", "tem_incidente_pai",
        "entrou_kpi_flag_fonte", "entrou_kpi_flag_calculada", "kpi_regra_divergente",
        "kpi_violado_fonte", "duracao_suspeita",
        "dt_aberto", "dt_resolvido", "dt_encerrado",
    )
)

(
    fato_incidentes.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(qualified_table(SCHEMA_GOLD, "fato_incidentes"))
)

print(f"gold.fato_incidentes gravada: {fato_incidentes.count()} linhas")

## Checagem de integridade referencial

Antes de considerar a Gold pronta, validamos que toda linha da fato tem
correspondência nas dimensões — nenhuma chave estrangeira "solta"
(o que quebraria os relacionamentos no Power BI).

In [ ]:
gold_fato = spark.table(qualified_table(SCHEMA_GOLD, "fato_incidentes"))
gold_dim_data = spark.table(qualified_table(SCHEMA_GOLD, "dim_data"))
gold_dim_produto = spark.table(qualified_table(SCHEMA_GOLD, "dim_produto"))

orfaos_data = gold_fato.join(gold_dim_data, "data_sk", "left_anti").count()
orfaos_produto = gold_fato.join(gold_dim_produto, "produto_sk", "left_anti").count()

print("Linhas da fato sem correspondência em dim_data:", orfaos_data)
print("Linhas da fato sem correspondência em dim_produto:", orfaos_produto)

assert orfaos_data == 0, "Integridade referencial quebrada em dim_data!"
assert orfaos_produto == 0, "Integridade referencial quebrada em dim_produto!"
print("Integridade referencial OK.")

## Fato agregada — `gold.fato_incidentes_diario`

Pré-agrega por (dia, produto, categoria, prioridade) — grão pensado
para alimentar diretamente os painéis do Power BI (Dashboard Principal e
Gráfico de Previsão de Incidentes, do protótipo da Sprint 2) via
DirectQuery, sem precisar agregar a fato granular a cada refresh.

`taxa_violacao_kpi` traz, no mesmo datamart, tanto o bruto
(`qtd_kpi_violado`) quanto a taxa — mesma decisão tomada nas tabelas de
risco da Silver (notebook `05`): taxa sozinha perde a noção de volume,
bruto sozinho distorce comparação entre segmentos de tamanhos
diferentes. `NULL` (não `0`) quando não há incidente elegível naquele
dia/segmento — mesma razão de lá: 0/0 é "sem dado", não "sem risco".

In [ ]:
fato_incidentes_diario = (
    silver_incidentes.filter(F.col("entrou_kpi_flag_fonte"))
    .groupBy("data_abertura", "produto", "categoria", "prioridade_num")
    .agg(
        F.count("*").alias("qtd_incidentes"),
        F.sum(F.when(F.col("kpi_violado_fonte"), 1).otherwise(0)).alias("qtd_kpi_violado"),
        F.round(F.avg("duracao_segundos"), 1).alias("duracao_media_segundos"),
    )
    .withColumn(
        "taxa_violacao_kpi",
        F.when(F.col("qtd_incidentes") > 0, F.round(F.col("qtd_kpi_violado") / F.col("qtd_incidentes"), 4)),
    )
    .withColumn("data_sk", F.date_format("data_abertura", "yyyyMMdd").cast("int"))
    .join(dim_produto, "produto", "left")
    .join(dim_categoria, "categoria", "left")
    .select(
        "data_sk", "produto_sk", "categoria_sk", "prioridade_num",
        "qtd_incidentes", "qtd_kpi_violado", "taxa_violacao_kpi", "duracao_media_segundos",
    )
)

(
    fato_incidentes_diario.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(qualified_table(SCHEMA_GOLD, "fato_incidentes_diario"))
)

print(f"gold.fato_incidentes_diario gravada: {fato_incidentes_diario.count()} linhas")

## Conferência final

In [ ]:
for t in ["dim_data", "dim_produto", "dim_categoria", "dim_equipe", "dim_prioridade", "fato_incidentes", "fato_incidentes_diario"]:
    full_name = qualified_table(SCHEMA_GOLD, t)
    print(f"{full_name}: {spark.table(full_name).count()} linhas")

display(spark.table(qualified_table(SCHEMA_GOLD, "fato_incidentes")).limit(5))